# 02-2 — extra test sets: Stanford Cars and ImageNet-1K val

A separate small notebook so the long ImageNet-100 training-set job of Step 2b does not have to run again.

**Before running (right panel):** GPU T4 · Internet On · *Add Input → Datasets:*
`rickyyyyyyy/torchvision-stanford-cars` (search the name in the Add Input box) and
*Competitions →* `imagenet-object-localization-challenge` (join the competition first) for ImageNet-1K val.

Outputs: `/kaggle/working/cache/cars.pt` and `imagenet.pt` → **Save Version (Save & Run All)**,
then attach this notebook's output as an input to the Step 3+4 notebook.

In [1]:
# 1) Paths, environment
import glob, os, sys, subprocess, pathlib
W = "/kaggle/working"; BASE = "/tmp/work"; CACHE = f"{W}/cache"
os.makedirs(BASE, exist_ok=True); os.makedirs(CACHE, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!ls /kaggle/input

Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
competitions  datasets


In [2]:
# 2) Dependencies, authors' code pinned to 50d7887, the two compatibility patches
!pip -q install ftfy regex git+https://github.com/openai/CLIP.git
os.chdir(BASE)
if not os.path.isdir("Defense-Prefix"):
    subprocess.run("git clone -q https://github.com/azuma164/Defense-Prefix.git", shell=True, check=True)
os.chdir(f"{BASE}/Defense-Prefix")
subprocess.run("git checkout -q 50d7887", shell=True, check=True)
p = pathlib.Path("datasets/utils/make_dataset_train.py"); s = p.read_text()
if "_textsize" not in s:
    helper = "\ndef _textsize(draw, text, font):\n    # Replacement for ImageDraw.textsize (removed in Pillow 10).\n    l, t, r, b = draw.multiline_textbbox((0, 0), text, font=font)\n    return r, b\n"
    s = s.replace("random.seed(300)", "random.seed(300)\n" + helper, 1).replace("imagedraw.textsize(text, font=font)", "_textsize(imagedraw, text, font)")
    p.write_text(s)
e = pathlib.Path("eval.py"); s = e.read_text()
e.write_text(s.replace("torch.load(args.tokens_path)", "torch.load(args.tokens_path, map_location=device, weights_only=False)"))
import torch, torchvision, PIL
print(subprocess.run("git rev-parse HEAD && git diff --stat", shell=True, capture_output=True, text=True).stdout)
print("python", sys.version.split()[0], "| torch", torch.__version__, "| torchvision", torchvision.__version__, "| Pillow", PIL.__version__)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
50d7887017b295e76b15e56c137ab47e68264448
 datasets/utils/make_dataset_train.py | 10 ++++++++--
 eval.py                              |  2 +-
 2 files changed, 9 insertions(+), 3 deletions(-)

python 3.13.16 | torch 2.11.0+cu130 | torchvision 0.26.0+cu130 | Pillow 13.0.0


In [3]:
%%writefile cache_features.py
"""Cache CLIP ViT-B/32 image features for one evaluation dataset of the Defense-Prefix paper.

Run from the root of the Defense-Prefix repository, one dataset per process
(the authors' attack generator seeds `random` once at import, and the authors
ran eval.py once per dataset, so one process per dataset reproduces their setup).

The dataset objects are the authors' own classes from `datasets/`, built with the
same arguments as the authors' `dataset.py`, so splits, labels, class names,
prompt templates and synthetic attacks are exactly theirs. The only difference
from `dataset.py` is that `download` is configurable (Stanford Cars can no longer
be downloaded through torchvision).

Features are computed exactly as in the authors' eval.py: fp16 CLIP image
features, L2-normalised. Evaluating any Defense-Prefix token on these cached
features gives the same predictions as eval.py.
"""
import argparse
import json
import os
import subprocess
import sys
import time

import clip
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

sys.path.insert(0, os.getcwd())

REAL_WORLD = {"rta-100", "paint", "disentangling"}
device = "cuda" if torch.cuda.is_available() else "cpu"


def build(name, preprocess, root="datasets", download=True):
    """Mirror of the authors' dataset.py with a configurable download flag."""
    if name == "imagenet":
        from datasets.ImageNetV2 import ImageNetValDataset
        return ImageNetValDataset(location=root, transform=preprocess)
    if name == "caltech":
        from datasets.caltech101 import Caltech101
        return Caltech101(root=root, transform=preprocess, download=download, make_typographic_dataset=True)
    if name == "pets":
        from datasets.oxford_pets import OxfordIIITPet
        return OxfordIIITPet(root=root, split="test", transform=preprocess, download=download, make_typographic_dataset=True)
    if name == "cars":
        from datasets.stanford_cars import StanfordCars
        return StanfordCars(root=root, split="test", transform=preprocess, download=False, make_typographic_dataset=True)
    if name == "flowers":
        from datasets.flowers102 import Flowers102
        return Flowers102(root=root, split="test", transform=preprocess, download=download, make_typographic_dataset=True)
    if name == "food":
        from datasets.food101 import Food101
        return Food101(root=root, split="test", transform=preprocess, download=download)
    if name == "aircraft":
        from datasets.fgvcaircraft import FGVCAircraft
        return FGVCAircraft(root=root, split="test", transform=preprocess, download=download)
    if name == "dtd":
        from datasets.dtd import DTD
        return DTD(root=root, split="test", transform=preprocess, download=download)
    if name == "eurosat":
        from datasets.eurosat import EuroSAT
        return EuroSAT(root=root, split="test", transform=preprocess, download=download)
    if name == "sun":
        from datasets.sun397 import SUN397
        return SUN397(root=root, split="test", transform=preprocess, download=False)
    if name == "paint":
        from datasets.paint_ta import PAINTDataset
        return PAINTDataset(root=root, transform=preprocess)
    if name == "disentangling":
        from datasets.disentangling_ta import DisentanglingDataset
        return DisentanglingDataset(root=root, transform=preprocess)
    if name == "rta-100":
        from datasets.rta100 import RTA100
        return RTA100(root=root, transform=preprocess)
    raise ValueError(name)


def git_commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    except Exception:
        return "unknown"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset", required=True)
    ap.add_argument("--out", required=True, help="folder for <dataset>.pt")
    ap.add_argument("--root", default="datasets")
    ap.add_argument("--batch_size", type=int, default=256)
    ap.add_argument("--workers", type=int, default=2)
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)

    model, preprocess = clip.load("ViT-B/32", device=device)
    model.eval()

    t0 = time.time()
    data = build(args.dataset, preprocess, root=args.root)  # also generates the synthetic attack images (seed 300)
    prep_s = time.time() - t0

    loader = DataLoader(data, batch_size=args.batch_size, shuffle=False, num_workers=args.workers)
    real = args.dataset in REAL_WORLD
    clean, attack, labels = [], [], []
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t1 = time.time()
    with torch.no_grad():
        for original_image, typographic_image, target in tqdm(loader, desc=args.dataset):
            f = model.encode_image(original_image.to(device))
            f /= f.norm(dim=-1, keepdim=True)          # identical to eval.py
            clean.append(f.cpu())
            if not real:
                g = model.encode_image(typographic_image.to(device))
                g /= g.norm(dim=-1, keepdim=True)
                attack.append(g.cpu())
            labels.append(torch.as_tensor(target))
    enc_s = time.time() - t1

    out = {
        "dataset": args.dataset,
        "real_world": real,
        "classes": list(data.classes),
        "template": data.templates[0],
        "labels": torch.cat(labels).long(),
        "clean": torch.cat(clean),                       # for real-world sets these images contain the attack
        "attack": torch.cat(attack) if attack else None,
        "meta": {
            "n_images": len(data),
            "n_classes": len(data.classes),
            "prepare_seconds": round(prep_s, 1),
            "encode_seconds": round(enc_s, 1),
            "peak_gpu_mem_mb": round(torch.cuda.max_memory_allocated() / 2**20, 1) if device == "cuda" else None,
            "gpu": torch.cuda.get_device_name(0) if device == "cuda" else "cpu",
            "torch": torch.__version__,
            "repo_commit": git_commit(),
            "model": "ViT-B/32",
        },
    }
    path = os.path.join(args.out, f"{args.dataset}.pt")
    torch.save(out, path)
    print(json.dumps({"saved": path, **out["meta"]}, indent=1))


if __name__ == "__main__":
    main()

Writing cache_features.py


In [4]:
%%writefile eval_cached.py
"""Evaluate CLIP and CLIP + Defense-Prefix on cached image features.

Text features and scoring follow the authors' eval.py exactly:
one template per dataset (templates[0]), "* {class}" with the * embedding
replaced by the DP vector, cosine similarity x 100, argmax.

Usage (from the Defense-Prefix repo root):
  python eval_cached.py --cache CACHE_DIR --token learned_token/dp_vit-b32.pt --tag authors_token
"""
import argparse
import csv
import glob
import json
import os
import sys

import clip
import torch
import torch.nn.functional as F

sys.path.insert(0, os.getcwd())
from utils.non_nv import encode_text_with_learnt_tokens  # authors' function

device = "cuda" if torch.cuda.is_available() else "cpu"

ORDER = ["imagenet", "caltech", "pets", "cars", "flowers", "food", "aircraft", "dtd", "sun", "eurosat",
         "disentangling", "paint", "rta-100"]
# Paper, Tables B, C and 3 (ViT-B/32): (CLIP, CLIP + DP)
PAPER = {
    "clean": {"imagenet": (62.02, 62.48), "caltech": (88.64, 89.28), "pets": (87.35, 87.22), "cars": (58.72, 57.47),
              "flowers": (66.32, 63.82), "food": (84.14, 83.65), "aircraft": (18.99, 19.26), "dtd": (44.57, 40.64),
              "sun": (61.74, 61.41), "eurosat": (42.98, 43.85)},
    "synthetic": {"imagenet": (39.10, 49.83), "caltech": (63.97, 79.54), "pets": (58.95, 72.88), "cars": (21.02, 28.64),
                  "flowers": (31.32, 44.12), "food": (56.27, 67.79), "aircraft": (10.83, 14.49), "dtd": (25.53, 31.6),
                  "sun": (34.02, 43.50), "eurosat": (4.86, 9.65)},
    "real": {"disentangling": (43.27, 71.93), "paint": (50.00, 63.64), "rta-100": (47.20, 58.00)},
}


def load_model():
    model, _ = clip.load("ViT-B/32", device=device)
    model.encode_text_with_learnt_tokens = type(model.encode_text)(encode_text_with_learnt_tokens, model)
    model.eval()
    return model


@torch.no_grad()
def class_text_features(model, classes, template, token=None):
    if token is None:
        toks = torch.cat([clip.tokenize(template.format(c)) for c in classes]).to(device)
        f = model.encode_text(toks)
        return f / f.norm(dim=-1, keepdim=True)
    toks = torch.cat([clip.tokenize(template.format(f"* {c}")) for c in classes]).to(device)
    star = clip.tokenize(["*"]).to(device)[0][1]
    f = model.encode_text_with_learnt_tokens(toks, star, token.unsqueeze(0), is_emb=False)
    return F.normalize(f, dim=-1)


@torch.no_grad()
def predictions(img_feats, text_feats, bs=8192):
    out = []
    for i in range(0, len(img_feats), bs):
        sim = 100.0 * (img_feats[i:i + bs].to(device) @ text_feats.T)
        out.append(sim.argmax(-1).cpu())
    return torch.cat(out)


def evaluate(model, cache_dir, token):
    """Returns rows: dataset, condition, n, acc_clip, acc_dp, plus per-image correctness for later stats."""
    rows, per_image = [], {}
    files = {os.path.basename(p)[:-3]: p for p in glob.glob(os.path.join(cache_dir, "*.pt"))}
    for name in [d for d in ORDER if d in files]:
        c = torch.load(files[name], map_location="cpu", weights_only=False)
        t_clip = class_text_features(model, c["classes"], c["template"])
        t_dp = class_text_features(model, c["classes"], c["template"], token)
        conds = [("real", c["clean"])] if c["real_world"] else [("clean", c["clean"]), ("synthetic", c["attack"])]
        for cond, feats in conds:
            y = c["labels"]
            ok_clip = predictions(feats, t_clip).eq(y)
            ok_dp = predictions(feats, t_dp).eq(y)
            per_image[f"{name}/{cond}"] = {"clip": ok_clip, "dp": ok_dp}
            rows.append({"dataset": name, "condition": cond, "n": len(y),
                         "acc_clip": round(100 * ok_clip.float().mean().item(), 2),
                         "acc_dp": round(100 * ok_dp.float().mean().item(), 2)})
    return rows, per_image


def with_paper(rows):
    for r in rows:
        p = PAPER.get(r["condition"], {}).get(r["dataset"])
        r["paper_clip"], r["paper_dp"] = (p if p else (None, None))
    return rows


def print_table(rows):
    print(f"{'dataset':<14}{'cond':<10}{'n':>7} | {'paper CLIP':>10}{'ours CLIP':>10} | {'paper DP':>9}{'ours DP':>9}")
    for r in rows:
        pc = "" if r["paper_clip"] is None else f"{r['paper_clip']:.2f}"
        pd = "" if r["paper_dp"] is None else f"{r['paper_dp']:.2f}"
        print(f"{r['dataset']:<14}{r['condition']:<10}{r['n']:>7} | {pc:>10}{r['acc_clip']:>10.2f} | {pd:>9}{r['acc_dp']:>9.2f}")
    for cond in ["clean", "synthetic", "real"]:
        sub = [r for r in rows if r["condition"] == cond]
        if sub:
            mc = sum(r["acc_clip"] for r in sub) / len(sub)
            md = sum(r["acc_dp"] for r in sub) / len(sub)
            print(f"  mean {cond:<9} over {len(sub)} sets: CLIP {mc:.2f} | DP {md:.2f}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--cache", required=True)
    ap.add_argument("--token", default="learned_token/dp_vit-b32.pt")
    ap.add_argument("--tag", default="authors_token")
    ap.add_argument("--out", default=None, help="results folder (default: <cache>/../results)")
    args = ap.parse_args()

    token = torch.load(args.token, map_location=device, weights_only=False)
    token = token.detach()
    if token.dim() == 1:          # authors save shape (1, 512); eval.py passes token.unsqueeze(0)
        token = token.unsqueeze(0)
    model = load_model()
    token = token.to(device).type(model.dtype)

    rows, per_image = evaluate(model, args.cache, token)
    rows = with_paper(rows)
    print_table(rows)

    out = args.out or os.path.join(os.path.dirname(os.path.abspath(args.cache)), "results")
    os.makedirs(out, exist_ok=True)
    with open(os.path.join(out, f"{args.tag}.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    torch.save(per_image, os.path.join(out, f"{args.tag}_per_image.pt"))
    print("saved", os.path.join(out, f"{args.tag}.csv"))


if __name__ == "__main__":
    main()

Writing eval_cached.py


## Stanford Cars (test split, 8,041 images)

In [ ]:
# What is actually attached? (Stanford Cars needs: cars_test/*.jpg, cars_test_annos_withlabels.mat, devkit/cars_meta.mat)
!ls /kaggle/input
print("== first levels")
!find -L /kaggle/input -maxdepth 4 -not -name "*.jpg" | head -60
print("== all .mat files")
!find -L /kaggle/input -maxdepth 8 -name "*.mat"
print("== number of jpg files per folder (top 10)")
!find -L /kaggle/input -maxdepth 6 -name "*.jpg" | xargs -n1 dirname | sort | uniq -c | sort -rn | head -10

In [5]:
# 3) The authors' StanfordCars class writes attacked images next to the data -> writable folder of symlinks
mat = glob.glob("/kaggle/input/**/cars_test_annos_withlabels.mat", recursive=True)
if not mat:
    mat = subprocess.run("find -L /kaggle/input -maxdepth 7 -name cars_test_annos_withlabels.mat", shell=True,
                         capture_output=True, text=True).stdout.split()
assert mat, "Stanford Cars input not found: add rickyyyyyyy/torchvision-stanford-cars in Add Input"
cars_src = os.path.dirname(mat[0]); print(cars_src, sorted(os.listdir(cars_src)))
CARS_ROOT = f"{BASE}/cars_root"; base = f"{CARS_ROOT}/stanford_cars"; os.makedirs(base, exist_ok=True)
for name in ["cars_test", "devkit", "cars_test_annos_withlabels.mat"]:
    s = f"{cars_src}/{name}"
    if os.path.isdir(f"{s}/{name}"): s = f"{s}/{name}"     # some copies nest cars_test/cars_test
    if not os.path.exists(f"{base}/{name}"): os.symlink(s, f"{base}/{name}")
!ls -la {base}; ls {base}/cars_test | wc -l    # expect 8041

/kaggle/input/datasets/rickyyyyyyy/torchvision-stanford-cars/stanford_cars ['cars_test', 'cars_test_annos_withlabels.mat', 'cars_train', 'devkit']
total 20
drwxr-xr-x 2 root root 4096 Nov 12 16:38 .
drwxr-xr-x 3 root root 4096 Nov 12 16:38 ..
lrwxrwxrwx 1 root root   84 Nov 12 16:38 cars_test -> /kaggle/input/datasets/rickyyyyyyy/torchvision-stanford-cars/stanford_cars/cars_test
lrwxrwxrwx 1 root root  105 Nov 12 16:38 cars_test_annos_withlabels.mat -> /kaggle/input/datasets/rickyyyyyyy/torchvision-stanford-cars/stanford_cars/cars_test_annos_withlabels.mat
lrwxrwxrwx 1 root root   81 Nov 12 16:38 devkit -> /kaggle/input/datasets/rickyyyyyyy/torchvision-stanford-cars/stanford_cars/devkit
8041


In [6]:
!python cache_features.py --dataset cars --root {CARS_ROOT} --out {CACHE} --workers 4 2>&1 | tail -12

{
 "saved": "/kaggle/working/cache/cars.pt",
 "n_images": 8041,
 "n_classes": 196,
 "prepare_seconds": 725.4,
 "encode_seconds": 51.0,
 "peak_gpu_mem_mb": 877.7,
 "gpu": "Tesla T4",
 "torch": "2.11.0+cu130",
 "repo_commit": "50d7887017b295e76b15e56c137ab47e68264448",
 "model": "ViT-B/32"
}


## ImageNet-1K val (50,000 images) — the paper's own ImageNet test set
Source: Kaggle competition **imagenet-object-localization-challenge** (ILSVRC 2012 data; join the competition once
to accept its rules, then *Add Input → Competitions*). The authors' download link (S3) is dead, so we rebuild the
folder their `ImageNetValDataset` class expects (`imagenetv2/imagenet_validation/<class>/<image>.JPEG`, 50,000 files)
from symlinks, with one folder per WordNet id. Sorted WordNet ids = the standard ImageNet class order = the order of
`datasets/configs/imagenet_config.py`, so the labels line up. The authors' own class then generates the attacks
(seed 300) and nothing else changes. Takes about 1 hour (attack generation is single-process, as in the authors' code).
**Sanity check:** zero-shot CLIP must give about 62% clean accuracy (paper: 62.02). A much lower number means the
label order is wrong.

In [7]:
# 4) Build the authors' expected layout from the Kaggle competition files (skips cleanly if the input is missing)
import csv
sol = glob.glob("/kaggle/input/**/LOC_val_solution.csv", recursive=True)
if not sol:
    print("ImageNet competition input not attached - skipping ImageNet-1K val")
else:
    comp = os.path.dirname(sol[0])
    val_dir = glob.glob(f"{comp}/**/Data/CLS-LOC/val", recursive=True)[0]
    mapping = [l.split()[0] for l in open(f"{comp}/LOC_synset_mapping.txt")]
    assert len(mapping) == 1000 and mapping == sorted(mapping), "synset mapping is not the sorted standard order"
    IN1K_ROOT = f"{BASE}/in1k_root"; dst = pathlib.Path(f"{IN1K_ROOT}/imagenetv2/imagenet_validation")
    if not dst.exists():
        for row in csv.DictReader(open(sol[0])):
            wnid = row["PredictionString"].split()[0]
            (dst / wnid).mkdir(parents=True, exist_ok=True)
            os.symlink(f"{val_dir}/{row['ImageId']}.JPEG", dst / wnid / f"{row['ImageId']}.JPEG")
    n_dirs = len(os.listdir(dst)); n_files = sum(len(os.listdir(dst / d)) for d in os.listdir(dst))
    print(n_dirs, "classes,", n_files, "images")    # expect 1000, 50000
    !python cache_features.py --dataset imagenet --root {IN1K_ROOT} --out {CACHE} --workers 4 2>&1 | grep -v -E "^(/|n[0-9]{8})" | tail -12

1000 classes, 50000 images
{
 "saved": "/kaggle/working/cache/imagenet.pt",
 "n_images": 50000,
 "n_classes": 1000,
 "prepare_seconds": 1019.5,
 "encode_seconds": 223.4,
 "peak_gpu_mem_mb": 877.7,
 "gpu": "Tesla T4",
 "torch": "2.11.0+cu130",
 "repo_commit": "50d7887017b295e76b15e56c137ab47e68264448",
 "model": "ViT-B/32"
}


In [8]:
# 5) Check with the authors' released token (compare with the paper: Cars and ImageNet)
!python eval_cached.py --cache {CACHE} --token learned_token/dp_vit-b32.pt --tag authors_token_2c --out {W}/results
!ls -la {CACHE}

dataset       cond            n | paper CLIP ours CLIP |  paper DP  ours DP
imagenet      clean       50000 |      62.02     62.04 |     62.48    62.52
imagenet      synthetic   50000 |      39.10     39.63 |     49.83    49.95
cars          clean        8041 |      58.72     58.66 |     57.47    57.43
cars          synthetic    8041 |      21.02     20.63 |     28.64    28.18
  mean clean     over 2 sets: CLIP 60.35 | DP 59.98
  mean synthetic over 2 sets: CLIP 30.13 | DP 39.06
saved /kaggle/working/results/authors_token_2c.csv
total 116580
drwxr-xr-x 2 root root      4096 Nov 12 17:57 .
drwxr-xr-x 5 root root      4096 Nov 12 16:10 ..
-rw-r--r-- 1 root root  16541318 Nov 12 16:51 cars.pt
-rw-r--r-- 1 root root 102822762 Nov 12 17:57 imagenet.pt


In [9]:
# Write a log per dataset (same role as the cache_<dataset>.log files of notebook 02-1)
import torch, json, os, glob
os.makedirs(f"{W}/logs", exist_ok=True)
for f in sorted(glob.glob(f"{CACHE}/*.pt")):
    d = torch.load(f, map_location="cpu", weights_only=False)
    name = os.path.basename(f)[:-3]
    with open(f"{W}/logs/cache_{name}.log", "w") as out:
        out.write(json.dumps({"saved": f, "dataset": d["dataset"], "template": d["template"], **d["meta"]}, indent=1))
    print(name, "->", f"{W}/logs/cache_{name}.log")
!python eval_cached.py --cache {CACHE} --token learned_token/dp_vit-b32.pt --tag authors_token_2c --out {W}/results > {W}/logs/eval_authors_token_2c.txt
!cat {W}/logs/eval_authors_token_02_2_.txt

cars -> /kaggle/working/logs/cache_cars.log
imagenet -> /kaggle/working/logs/cache_imagenet.log
cat: /kaggle/working/logs/eval_authors_token_02_2_.txt: No such file or directory
